# Drone power draw prediction via flight sensor information 

In [1]:
!uv pip install numpy pandas scipy matplotlib seaborn scikit-learn statsmodels jupyter ipykernel

Using Python 3.14.7 environment at: D:\PC DISAINE\toky\ALU\term_6\math_for_ml\summative\datasets\drone_power_draw\power_draw_regression\summative\.venv
Checked 9 packages in 32ms


## Task 1.1: Data preparation and exploration

In [2]:
import urllib.request
import zipfile

# The raw data is not in the repo (94 MB), so fetch it from KiltHub on the first run.
# File IDs come from https://api.figshare.com/v2/articles/12683453
KILTHUB = "https://ndownloader.figshare.com/files/"

if not PARAMETERS_PATH.exists() or not FLIGHTS_DIR.exists():
    DATA_DIR.mkdir(exist_ok=True)
    urllib.request.urlretrieve(KILTHUB + "26385034", PARAMETERS_PATH)

    zip_path = DATA_DIR / "flights.zip"                  # about 41 MB
    urllib.request.urlretrieve(KILTHUB + "26385070", zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)                           # creates data/flights/1.csv ... 279.csv
    zip_path.unlink()
    print("Downloaded the dataset into", DATA_DIR)

NameError: name 'PARAMETERS_PATH' is not defined

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Project root directory
PROJECT_ROOT = Path.cwd().parent.parent

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
FLIGHTS_DIR = DATA_DIR / "flights"
PARAMETERS_PATH = DATA_DIR / "parameters.csv"

# print("Project root:", PROJECT_ROOT)
# print("Data directory:", DATA_DIR)

In [ ]:
def load_dataset(flights_dir, parameters_path):
    """
    Load per-flight sensor logs and the flight-level parameters.

    Returns:
        flights_df: All flight observations stacked, with a `flight` ID column.
        parameters_df: One row per flight (speed, payload, altitude, ...).
    """
    flights_dir = Path(flights_dir)

    parameters_df = pd.read_csv(parameters_path)

    # Only numeric filenames (e.g. 12.csv), sorted numerically, not alphabetically
    flight_files = sorted(
        (f for f in flights_dir.glob("*.csv") if f.stem.isdigit()),
        key=lambda f: int(f.stem),
    )
    if not flight_files:
        raise FileNotFoundError(f"No flight CSV files found in {flights_dir}")

    flights_df = pd.concat(
        (pd.read_csv(f).assign(flight=int(f.stem)) for f in flight_files),
        ignore_index=True,
    )

    return flights_df, parameters_df


flights_df, parameters_df = load_dataset(FLIGHTS_DIR, PARAMETERS_PATH)

In [ ]:
print(f"Loaded {flights_df['flight'].nunique()} flights")
print(f"Combined dataset shape: {flights_df.shape}")

In [ ]:
# Every flight log should have a parameters row, and vice versa
log_ids = set(flights_df["flight"])
param_ids = set(parameters_df["flight"])
print("Logs without parameters:", sorted(log_ids - param_ids))
print("Parameters without logs:", sorted(param_ids - log_ids))

assert parameters_df["flight"].is_unique, "Duplicate flight IDs in parameters.csv"

data_df = flights_df.merge(parameters_df, on="flight", how="left", validate="many_to_one")

print(f"Loaded {flights_df['flight'].nunique()} flights")
print(f"Flight logs shape: {flights_df.shape}")
print(f"Merged dataset shape: {data_df.shape}")

In [ ]:
data_df.info()

display(data_df.describe().T)  # transposed: easier to read with ~25 columns

missing = data_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print("No missing values.")
else:
    display(missing.to_frame("missing").assign(pct=lambda d: 100 * d["missing"] / len(data_df)))

# Observations per flight
rows_per_flight = data_df["flight"].value_counts().sort_index()
display(rows_per_flight.describe())

### Target definition: power in watts

We can notice two apparent clusters in power, one near 0W and the second around 450-550W. The mean wattage per flight falls under a slightly smaller range, this could clue us that most variations happens within a flight. We might add a `flight_phase` feature later to capture the different flight state neatly.

In [ ]:
data_df["power"] = data_df["battery_voltage"] * data_df["battery_current"]

display(data_df["power"].describe())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

data_df["power"].hist(bins=100, ax=ax[0]); ax[0].set_title("Power (W)")
data_df.groupby("flight")["power"].mean().hist(bins=40, ax=ax[1]); ax[1].set_title("Mean power per flight")
plt.show()

### Data quality checks

In [ ]:
print("Negative current rows:", (data_df["battery_current"] < 0).sum())

for c in ["position_x", "position_y", "position_z"]:
    print(c, "== 0 rows:", (data_df[c] == 0).sum(), "| flights:", data_df.loc[data_df[c] == 0, "flight"].nunique())
    
q = data_df[["orientation_x","orientation_y","orientation_z","orientation_w"]]

print("Quaternion norm range:", np.linalg.norm(q, axis=1).min(), np.linalg.norm(q, axis=1).max())
print("Duplicated rows:", data_df.duplicated().sum())

display(data_df["altitude"].value_counts())

dt = data_df.groupby("flight")["time"].diff()
display(dt.describe())   # sampling interval: should be about 0.2 s

### Flight-level parameters (one row per flight)

In [ ]:
display(pd.crosstab(parameters_df["speed"], parameters_df["payload"]))
display(pd.crosstab(parameters_df["route"], parameters_df["altitude"]))

### Power over time for a few flights

In [ ]:
import seaborn as sns

sample = parameters_df.groupby("payload")["flight"].first().tolist()
fig, axes = plt.subplots(len(sample), 1, figsize=(12, 3*len(sample)), sharex=True)

for ax, f in zip(axes, sample):
    d = data_df[data_df["flight"] == f]
    ax.plot(d["time"], d["power"], label="power")
    ax2 = ax.twinx()
    ax2.plot(d["time"], d["position_z"], color="orange", alpha=.6, label="altitude (position_z)")
    ax.set_title(f"Flight {f}: payload {d['payload'].iat[0]} g, speed {d['speed'].iat[0]}")

    # Combine the lines from both axes into one legend
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper right")

plt.tight_layout(); plt.show()

The power is idle at first then jumps during takeoff, and drops to about 0 after the landing. We can see that the power follows the altitude curve. This tells us that the flight phase and height matters a lot for our predictions.

### Power against each predictor

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=data_df, x="payload", y="power", ax=axes[0])
sns.boxplot(data=data_df, x="speed", y="power", ax=axes[1])
sns.boxplot(data=data_df, x="altitude", y="power", ax=axes[2])
plt.show()

The median power rises as the payload gets heavier: 0 -> 250 -> 500 g (row means 365 -> 408 -> 440 W). Only one flight uses a 750g payload, so we cannot read useful trends from it. In addition, speed doesn't give off any useful trend (r = −0.03) and altitude shows very small fluctuations (387–421 W).

**Decision**: keep payload numeric; keep speed despite its weak link for later feature engineering; treat altitude as labels.

### Correlation and multicollinearity

In [ ]:
num = data_df.select_dtypes("number").drop(columns=["flight"])
plt.figure(figsize=(14, 11))
sns.heatmap(num.corr(), cmap="coolwarm", center=0, annot=False); plt.show()
display(num.corr()["power"].drop("power").sort_values(key=abs, ascending=False))

We can observe a near perfect correlation between `battery_current` and `power` (r = 0.997), that's because power is computed from **voltage * current**. The heatmap also shows high correlation between `position_x` and `position_y`, but that comes from the 9 flights with (0, 0) GPS readings. Without considering the aforementioned correlations, the strongest links are `position_z` (0.50), `wind_speed`(0.48) and `velocity_y` (0.34).

**Decision**: drop current and voltage as they leak the target, drop raw GPS, and use height for the scatter plots before and after the fit.

## Task 1.2: Feature engineering

### Remove flights with weird properties

In [ ]:
# Flight 211 to 219, routes A1-A3 have their GPS logged as exactly 0, it is most likely a sensor failure, not real positions
bad_gps = data_df.loc[(data_df["position_x"] == 0) & (data_df["position_y"] == 0), "flight"].unique()

before = len(data_df)
data_df = data_df[~data_df["flight"].isin(bad_gps)].copy()
print(f"Rows: {before:,} -> {len(data_df):,} | flights left: {data_df['flight'].nunique()}")

display(data_df.groupby("route").agg(flights=("flight", "nunique"), rows=("flight", "size"))
        .sort_values("rows", ascending=False))

### Engineer new numerical features

In [ ]:
data_df["ground_speed"] = np.hypot(data_df["velocity_x"], data_df["velocity_y"])

qx, qy = data_df["orientation_x"], data_df["orientation_y"]
data_df["tilt_deg"] = np.degrees(np.arccos(np.clip(1 - 2 * (qx**2 + qy**2), -1, 1)))

data_df["height"] = data_df["position_z"] - data_df.groupby("flight")["position_z"].transform("first")

### Categorical feature 1: flight phase

In [ ]:
# Motor vibration: rolling std (about 1 s) of the acceleration magnitude.
# Motors off -> about 0.001; motors on -> about 0.16 (median).
acc = data_df[["linear_acceleration_x", "linear_acceleration_y", "linear_acceleration_z"]].to_numpy()
data_df["acc_norm"] = np.linalg.norm(acc, axis=1)
data_df["vibration"] = (data_df.groupby("flight")["acc_norm"]
                        .transform(lambda s: s.rolling(5, center=True, min_periods=1).std()))

MOTORS_OFF = 0.02 # threshold chosen from the vibration distribution
vz, gs = data_df["velocity_z"], data_df["ground_speed"]

# Checked in order: the first condition that is true sets the label
data_df["flight_phase"] = np.select(
    [data_df["vibration"] < MOTORS_OFF, vz > 0.5, vz < -0.5, gs > 1.0],
    ["ground", "climb",  "descent",  "cruise"],
    default="hover",
)

display(data_df.groupby("flight_phase")["power"].agg(rows="size", mean_W="mean", std_W="std").round(1))

### Categorical features 2 and 3: route and altitude

In [ ]:
data_df["route"] = np.where(data_df["route"].isin(["R1", "H"]), data_df["route"], "other")
data_df["altitude"] = data_df["altitude"].astype(str)   # "25","50","75","100","3","25-50-100-25"

display(data_df.groupby("route")["flight"].nunique())

### Drop the columns we will not use

Check the comments for the reasons

In [ ]:
DROP_COLS = [
    "date", "local_time",                              # calendar/clock time with no physical effect
    "position_x", "position_y", "position_z",          # GPS longitude and latitude which will be replaced by height
    "orientation_x", "orientation_y",
    "orientation_z", "orientation_w",                  # quaternion, replaced by tilt_deg
    "velocity_x", "velocity_y",                        # direction-dependent, replaced by ground_speed
    "angular_x", "angular_y", "angular_z",             # noisy spin rates, |r| < 0.07
    "linear_acceleration_x", "linear_acceleration_y",
    "linear_acceleration_z", "acc_norm", "vibration",  # only used to label flight_phase
    "wind_angle",                                      # |r| = 0.08, measured in the drone's own frame
]
data_df = data_df.drop(columns=DROP_COLS)

### Final feature list

In [ ]:
NUMERIC_FEATURES = ["payload", "speed", "wind_speed", "ground_speed",
                    "velocity_z", "tilt_deg", "height", "time"]
CATEGORICAL_FEATURES = ["route", "flight_phase", "altitude"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES      # 11 inputs
TARGET = "power"

display(data_df[NUMERIC_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET)
        .sort_values(key=abs, ascending=False).to_frame("r with power"))
display(data_df[NUMERIC_FEATURES].agg(["min", "max"]).T)   # use these for the Pydantic ranges

### Split the data

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# One fold of 5 = about 20% test; this seed puts every route group (R1, H, other) in the test set
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=5)
train_idx, test_idx = next(sgkf.split(data_df, y=data_df["route"], groups=data_df["flight"]))

train_df, test_df = data_df.iloc[train_idx], data_df.iloc[test_idx]
assert set(train_df["flight"]).isdisjoint(test_df["flight"])

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]
print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")

### Final list of inputs

| Field | Type | Unit | Meaning | Observed (min – max) | 99% of rows | **Allowed** |
|---|---|---|---|---|---|---|
| `payload` | float | g | Mass of the parcel carried | 0 – 750 (only 0, 250, 500, 750) | 0 – 500 | **0 – 750** |
| `speed` | float | m/s | Programmed cruise ground speed | 0 – 12 (only 0, 4, 6, 8, 10, 12) | 0 – 12 | **0 – 12** |
| `wind_speed` | float | m/s | Wind speed from the onboard anemometer | 0 – 18.1 | 0.2 – 13.9 | **0 – 20** |
| `ground_speed` | float | m/s | Horizontal speed, √(vx² + vy²) | 0 – 12.7 | 0 – 11.5 | **0 – 13** |
| `velocity_z` | float | m/s | Vertical speed (+ up, − down) | −4.0 – 5.8 | −3.1 – 3.6 | **−5 – 6** |
| `tilt_deg` | float | ° | Tilt from level, from the quaternion | 0 – 67.1 | 0.3 – 23.3 | **0 – 45** |
| `height` | float | m | Height above the take-off point | −13.8 – 107.7 | −4.9 – 102.0 | **−15 – 120** |
| `time` | float | s | Seconds since logging started | 0 – 428.3 | 1.8 – 252.2 | **0 – 450** |


### Categorical inputs

| Field | Type | Allowed values | Meaning |
|---|---|---|---|
| `route` | string enum | `"R1"`, `"H"`, `"other"` | R1 is the main delivery route (182 flights). H is hover-only. `other` groups routes R2–R7. |
| `flight_phase` | string enum | `"ground"`, `"climb"`, `"cruise"`, `"hover"`, `"descent"` | What the drone is doing at that moment |
| `altitude` | string enum | `"3"`, `"25"`, `"50"`, `"75"`, `"100"`, `"25-50-100-25"` | Programmed cruise altitude in m. It is a string because of the multi-level profile. |


### Target (only in `/retrain`)

| Field | Type | Unit | Observed | **Allowed** |
|---|---|---|---|---|
| `power` | float | W | −7.7 – 1004.2 (99%: −3.1 – 743.9) | **−10 – 1100** |

## Task 2: Modelling

### Task 2.0: Create a dummy model for API and app development

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sample = train_df.sample(2000, random_state=0)

placeholder = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])),
    ("model", LinearRegression()),
])
placeholder.fit(sample[FEATURES], sample[TARGET])

joblib.dump(placeholder, "../API/best_model.pkl")

loaded = joblib.load("../API/best_model.pkl")
print(loaded.predict(X_test.iloc[[0]]), y_test.iloc[0])

### Task 2.1: Train the real models

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor


def make_pipeline(model):
    """Same preprocessing for every model: scale the numbers, one-hot the labels."""
    return Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", StandardScaler(), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ])),
        ("model", model),
    ])


# max_depth=12 for both trees: deeper trees memorise the training flights
# (an unlimited tree reaches train MSE 0 but test MSE ~8,000)
models = {
    "SGD": make_pipeline(SGDRegressor(random_state=0)),
    "Linear regression": make_pipeline(LinearRegression()),
    "Decision tree": make_pipeline(DecisionTreeRegressor(max_depth=12, random_state=0)),
    "Random forest": make_pipeline(RandomForestRegressor(n_estimators=50, max_depth=12,
                                                         n_jobs=-1, random_state=0)),
}

rows = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    rows.append({
        "model": name,
        "train MSE": round(mean_squared_error(y_train, pipe.predict(X_train))),
        "test MSE": round(mean_squared_error(y_test, pipe.predict(X_test))),
    })

results = pd.DataFrame(rows).set_index("model")
display(results)

### Task 2.2: Tune the SGD

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning


def sgd_mse(**params):
    """Fit SGD with these settings (the rest stay default); return train and test MSE."""
    pipe = make_pipeline(SGDRegressor(random_state=0, **params)).fit(X_train, y_train)
    return (mean_squared_error(y_train, pipe.predict(X_train)),
            mean_squared_error(y_test, pipe.predict(X_test)))


# Change one setting at a time; defaults are eta0=0.01, alpha=0.0001, max_iter=1000
experiments = {
    "eta0": [0.00001, 0.0001, 0.001, 0.01, 0.1, 0.5],   # learning rate
    "alpha": [0.00001, 0.0001, 0.001, 0.01, 0.1],       # L2 regularization strength
    "max_iter": [1, 2, 5, 10, 20, 50],                  # epochs
}

rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)  # expected with few epochs
    for setting, values in experiments.items():
        for value in values:
            # tol=None runs exactly max_iter epochs instead of stopping early
            extra = {"tol": None} if setting == "max_iter" else {}
            train_mse, test_mse = sgd_mse(**{setting: value}, **extra)
            rows.append({"setting": setting, "value": str(value),
                         "train MSE": round(train_mse), "test MSE": round(test_mse)})

display(pd.DataFrame(rows))

- **Learning rate (`eta0`)**: a small learning rate (0.00001) will reach the minimum after an ungodly amount of time (it didn't reach it after 1000 epochs). A large learning rate (0.5) just jumps around the minimum without ever settling. The optimal range is 0.0001–0.01: the default returned the lowest test MSE (5,127).
- **Regularization (`alpha`)**: 0.001 barely does anything; 0.01 and higher shrinks the coefficients too much, leading to underfitting. The train and test MSE rise together, reaching 9,815 at 0.1. The default 0.0001 wins.
- **Epochs (`max_iter`)**: one epoch is approximately 200,000 weight updates, so 1 epoch almost converges (5,129). The default settings stops at 10 epochs, any longer and the MSE rises again because the weights keep revolving around the minimum.

**Decision**: keep the default hyperparameters

### Task 2.3: Graph the loss curve

In [ ]:
# Regurgitate the preprocessing above 
preprocess = models["SGD"][:-1]
Xp_train, Xp_test = preprocess.transform(X_train), preprocess.transform(X_test)

# Same settings as the tuned SGD; each partial_fit call is one pass (epoch) over the training rows
sgd = SGDRegressor(random_state=0)
EPOCHS = 30
train_loss, test_loss = [], []
for epoch in range(EPOCHS):
    sgd.partial_fit(Xp_train, y_train)
    train_loss.append(mean_squared_error(y_train, sgd.predict(Xp_train)))
    test_loss.append(mean_squared_error(y_test, sgd.predict(Xp_test)))

epochs = range(1, EPOCHS + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, train_loss, marker="o", label="train MSE")
plt.plot(epochs, test_loss, marker="o", label="test MSE")
plt.xlabel("Epoch"); plt.ylabel("MSE (W$^2$)"); plt.title("SGD loss per epoch")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Task 2.4: Scatter plot before and after the fit

In [ ]:
# Numeric feature with the strongest correlation to power (on the training rows)
top = train_df[NUMERIC_FEATURES].corrwith(train_df[TARGET]).abs().idxmax()

# One-feature linear regression: power = slope * top + intercept
line = LinearRegression().fit(train_df[[top]], train_df[TARGET])
x_line = pd.DataFrame({top: np.linspace(train_df[top].min(), train_df[top].max(), 100)})
test_mse = mean_squared_error(y_test, line.predict(test_df[[top]]))
print(f"Top feature: {top} | power = {line.coef_[0]:.2f} * {top} + {line.intercept_:.1f}"
      f" | test MSE: {test_mse:,.0f}")

sample = train_df.sample(5000, random_state=0)  # 5,000 points keep the plot readable
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, title in zip(axes, ["Before: raw data", "After: one-feature regression line"]):
    ax.scatter(sample[top], sample[TARGET], s=5, alpha=0.3)
    ax.set_xlabel(top); ax.set_title(title)
axes[0].set_ylabel("power (W)")
axes[1].plot(x_line[top], line.predict(x_line), color="red", lw=2, label="fitted line")
axes[1].legend()
plt.show()

### Task 2.5: Model comparison and selection

In [ ]:
# Rank the models scored in Task 2.1 (tuning kept SGD's defaults, so those scores still hold), best first
ranking = results.sort_values("test MSE")

best_name = ranking.index[0]
best_model = models[best_name]

print(f"Best model (lowest test MSE): {best_name}")

**Best model: Random forest, with a test MSE of 4,062**

**SGD and Linear regression** returned very similar scores. They are both straight line models that reaches the same minimum, but they cannot capture the wattage jump from ground (~0 W) to flight (~0 W) beyond what information the `flight_phase` labels give them.

**Decision tree** has a low train MSE (2,409) but the worst test MSE (5,472). The tree learns the training flights too closely but does not generalize well for new observations. 

**Random forest** also performs worse on test data than train data (2,171 vs 4,062) but the 50 trees cancels most of the overfitting of individual trees, thus giving the lowest test MSE score of the 4 models.

### Task 2.6: Save the winning model



In [ ]:
MODEL_PATH = "../API/best_model.pkl"
TRAIN_DATA_PATH = "../API/train_data.csv.gz"

# The whole Pipeline is saved, so the API gets the scaler, the encoder and the model in one file
joblib.dump(best_model, MODEL_PATH, compress=3)

# /retrain refits on these rows plus the uploaded ones, so it needs the same columns.
# mtime=0 keeps the file identical on every rerun, so git only sees real changes.
train_df[FEATURES + [TARGET]].to_csv(
    TRAIN_DATA_PATH, index=False, compression={"method": "gzip", "mtime": 0}
)
print(f"Saved {len(train_df):,} training rows to {TRAIN_DATA_PATH}")

print(f"Saved {best_name} to {MODEL_PATH}")


def predict_power(row, model_path=MODEL_PATH):
    """Load the saved Pipeline and predict the power (W) of one row of inputs."""
    model = joblib.load(model_path)
    return float(model.predict(row[FEATURES])[0])


row = X_test.iloc[[0]]  # double brackets keep a one-row DataFrame
print(f"Predicted: {predict_power(row):.1f} W | actual: {y_test.iloc[0]:.1f} W")